# Studi Kasus 4 — Hybrid RAG

BM25 menangkap exact term dan dense retrieval menangkap kemiripan semantik. Reciprocal Rank Fusion menggabungkan posisi ranking tanpa mengharuskan kedua skor berada pada skala sama.


In [ ]:
!pip -q install -U rank-bm25 sentence-transformers pandas


In [ ]:
import re, numpy as np, pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

docs=[
 "XLM-RoBERTa XNLI dapat dipakai untuk zero-shot classification multilingual.",
 "LoRA melatih matriks low-rank sambil membekukan bobot dasar.",
 "QLoRA menggabungkan quantization 4-bit dan adapter LoRA.",
 "RAG mengambil konteks relevan sebelum model generatif menjawab.",
 "BM25 menggunakan kecocokan term dan inverse document frequency."
]
tokenize=lambda x:re.findall(r'\w+',x.lower())
bm25=BM25Okapi([tokenize(d) for d in docs])
encoder=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
doc_emb=encoder.encode(docs,normalize_embeddings=True)


## Sparse, dense, dan Reciprocal Rank Fusion


In [ ]:
def ranks_desc(scores):
    order=np.argsort(scores)[::-1]
    return {int(doc_id):rank+1 for rank,doc_id in enumerate(order)}

def hybrid_search(query,k=60,top_n=4):
    sparse=np.array(bm25.get_scores(tokenize(query)))
    q=encoder.encode(query,normalize_embeddings=True)
    dense=doc_emb@q
    sr,dr=ranks_desc(sparse),ranks_desc(dense)
    fused=np.array([1/(k+sr[i])+1/(k+dr[i]) for i in range(len(docs))])
    order=np.argsort(fused)[::-1][:top_n]
    return [{"id":int(i),"document":docs[i],"bm25":float(sparse[i]),
             "dense":float(dense[i]),"rrf":float(fused[i])} for i in order]

pd.DataFrame(hybrid_search("bagaimana fine tuning model dengan bobot 4 bit"))


## Optional cross-encoder reranking

Reranker membaca query dan document bersama sehingga lebih akurat, tetapi jauh lebih lambat daripada bi-encoder.


In [ ]:
reranker=CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

def search_and_rerank(query,top_n=3):
    candidates=hybrid_search(query,top_n=5)
    scores=reranker.predict([(query,x["document"]) for x in candidates])
    for x,s in zip(candidates,scores):x["rerank_score"]=float(s)
    return sorted(candidates,key=lambda x:x["rerank_score"],reverse=True)[:top_n]

pd.DataFrame(search_and_rerank("apa itu adaptasi low rank"))


## Evaluasi

Bandingkan BM25-only, dense-only, RRF, dan RRF+reranker. Gunakan query exact term, parafrasa, singkatan, serta hard negative. Laporkan Recall@k, MRR, nDCG@k, latency per tahap, dan reranking gain.
